
# Darukaa Adaptive Biodiversity Assessment — Nandoshi Lake

**Profile:** `aquatic_lake`  
**Year-0 baseline:** 1 Aug 2025–31 Aug 2026  
**Historical trend/context:** 2018–2026

This notebook is designed to run **cell-by-cell in Google Colab**. The repository setup cell is safe to rerun: it pulls the latest `main` commit when the repository already exists, reinstalls the local adaptive package, and clears cached Python imports so manual GitHub edits are picked up.

> **Important:** the supplied KML is treated as the fixed master assessment boundary. Water extent is derived dynamically from EO. No secondary water polygon is manually required.


## Run order

Run top-to-bottom. For later GitHub updates, rerun the **repository sync/install** cell before rerunning analysis cells.

This notebook uses the common Darukaa scoring architecture:
`raw value → reference comparison → intactness (0–100) → indicator concern → C1/C2/C3/C4 pillar geometric mean → overall SoN geometric mean`.

The five concern bands are a declared Darukaa product convention:
**80–100 Very Low, 60–<80 Low, 40–<60 Moderate, 20–<40 High, 0–<20 Very High**.


In [ ]:

# 0. Runtime configuration
import os, sys, subprocess, importlib
from pathlib import Path

REPO_URL = "https://github.com/G-auravSingh/reference-benchmarking.git"
BRANCH = "main"
REPO_DIR = Path("/content/reference-benchmarking")
PACKAGE_DIR = REPO_DIR / "darukaa_adaptive_v1.0.0"
PROFILE_PATH = PACKAGE_DIR / "profiles" / "aquatic_lake.yaml"
GEE_PROJECT = "gaurav-singh-007"  # change only when using a different GEE project
PULL_LATEST = True

print("Repository:", REPO_URL)
print("Adaptive package:", PACKAGE_DIR)
print("Profile:", PROFILE_PATH)


In [ ]:

# 1. Sync GitHub and install the adaptive package.
# Safe for repeated runs: fast-forward-only pull; never resets/discards local changes.

def run_cmd(cmd, cwd=None):
    print("$", " ".join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout.rstrip())
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr.rstrip())
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result

if not REPO_DIR.exists():
    run_cmd(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)])
elif PULL_LATEST:
    status = subprocess.run(["git", "status", "--porcelain"], cwd=REPO_DIR, text=True, capture_output=True)
    if status.stdout.strip():
        raise RuntimeError(
            "The repository has local uncommitted changes. Commit/stash them before pulling; "
            "the notebook will not discard them automatically."
        )
    run_cmd(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO_DIR)

# Make the current package directory take precedence over any older installed copy.
sys.path.insert(0, str(PACKAGE_DIR))
run_cmd([sys.executable, "-m", "pip", "install", "-q", "-r", str(PACKAGE_DIR / "requirements.txt")])
run_cmd([sys.executable, "-m", "pip", "install", "-q", "-e", str(PACKAGE_DIR)])

for name in list(sys.modules):
    if name == "darukaa_adaptive" or name.startswith("darukaa_adaptive."):
        del sys.modules[name]
importlib.invalidate_caches()

GIT_SHA = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("✓ Adaptive package installed")
print("✓ Git commit:", GIT_SHA)


## 2. Google Earth Engine authentication

In [ ]:

import ee

try:
    ee.Initialize(project=GEE_PROJECT)
    print("✓ Earth Engine initialized:", GEE_PROJECT)
except Exception as exc:
    print("Earth Engine authentication required:", exc)
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)
    print("✓ Earth Engine initialized:", GEE_PROJECT)


In [ ]:

# 3. Confirm the expected Earth Engine datasets are addressable.
checks = {
    "Dynamic World": "GOOGLE/DYNAMICWORLD/V1",
    "Sentinel-2 SR Harmonized": "COPERNICUS/S2_SR_HARMONIZED",
    "Sentinel-1 GRD": "COPERNICUS/S1_GRD",
}
for label, asset in checks.items():
    try:
        count = ee.ImageCollection(asset).limit(1).size().getInfo()
        print(f"✓ {label}: {asset} (collection accessible; sample size={count})")
    except Exception as exc:
        print(f"✗ {label}: {asset} -> {exc}")


## 3. Upload the master project KML/KMZ

In [ ]:

from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("Upload the master project KML/KMZ to continue.")

site_candidates = [Path(name) for name in uploaded if Path(name).suffix.lower() in {".kml", ".kmz"}]
if not site_candidates:
    raise ValueError("No .kml or .kmz file was uploaded.")
SITE_FILE = str(site_candidates[0])
print("✓ Master assessment boundary:", SITE_FILE)


## 4. Load configuration and run geometry QA

In [ ]:

from darukaa_adaptive.config import AssessmentConfig
from darukaa_adaptive.site import read_kml, area_ha, geometry_hash, validate_site_geometry, make_shapely_domains, ee_geometry

cfg = AssessmentConfig.from_yaml(PROFILE_PATH)
cfg.gee.project_id = GEE_PROJECT
errors = cfg.validate()
if errors:
    raise ValueError(errors)

geom, named_parts = read_kml(SITE_FILE)
qa = validate_site_geometry(geom)
print("Configuration baseline:", cfg.temporal.baseline_start_date, "to", cfg.temporal.baseline_end_date)
print("Baseline label:", cfg.temporal.baseline_label)
print("Master boundary area (ha):", round(area_ha(geom), 6))
print("Named KML parts:", list(named_parts)[:20])
print("Geometry SHA-256:", geometry_hash(SITE_FILE))
print("Geometry QA:", qa)


In [ ]:

# 5. Create deterministic fixed domains.
from darukaa_adaptive.site import make_domains

domains = make_domains(geom, cfg.spatial.riparian_buffer_m, cfg.spatial.context_buffer_km)
print("✓ Master boundary")
print(f"✓ Fixed riparian ring: {cfg.spatial.riparian_buffer_m:.0f} m")
print(f"✓ Context ring: {cfg.spatial.context_buffer_km:.1f} km")
print("Dynamic water is generated from EO; no manual secondary water polygon is needed.")


In [ ]:

# 6. Interactive map: inspect the fixed spatial framework.
import geemap

m = geemap.Map()
m.centerObject(domains["boundary"], 14)
m.addLayer(domains["boundary"], {"color": "red"}, "Master boundary")
m.addLayer(domains["riparian_fixed"], {"color": "yellow"}, "Fixed riparian 100 m")
m.addLayer(domains["context"], {"color": "blue"}, "Context ring")
m


## 5. Year-0 dynamic-water diagnostics

In [ ]:

from darukaa_adaptive.water import WaterDetector
from darukaa_adaptive.periods import month_periods

water = WaterDetector(cfg)
BASELINE_START, BASELINE_END = cfg.temporal.baseline_dates()
summary = water.area_summary(domains["boundary"], BASELINE_START, BASELINE_END)
print(summary)


In [ ]:

# 7. Monthly water series spanning the complete Year-0 seasonal cycle.
import pandas as pd

monthly_period_list = month_periods(
    cfg.temporal.baseline_start_date.isoformat(),
    cfg.temporal.baseline_end_date.isoformat()
)
monthly = water.period_metrics(domains["boundary"], monthly_period_list)
monthly_df = pd.DataFrame(monthly)
monthly_df[["period_start","period_end_inclusive","water_area_ha","water_fraction_pct","occurrence_fraction","method","status"]]


In [ ]:

# 8. View the dynamically generated water mask for a selected baseline month.
selected = monthly_period_list[len(monthly_period_list)//2]
mask, method, n_images = water.water_mask_for_period(domains["boundary"], selected["start"], selected["end"])
print("Selected period:", selected, "method:", method, "images:", n_images)

m2 = geemap.Map()
m2.centerObject(domains["boundary"], 14)
m2.addLayer(domains["boundary"], {"color": "red"}, "Master boundary")
m2.addLayer(mask.selfMask(), {"palette": ["0000FF"]}, "EO-derived water mask")
m2


## 6. Run the lake metric suite

In [ ]:

from darukaa_adaptive.metrics import LakeMetrics
from darukaa_adaptive.qa import qa_metrics

metrics_engine = LakeMetrics(cfg, water)
metric_results = metrics_engine.run(
    domains["boundary"],
    domains["riparian_fixed"],
    BASELINE_START,
    BASELINE_END,
)

metric_df = pd.DataFrame([m.to_dict() for m in metric_results])
metric_df[[
    "metric","pillar","domain","value","units","status","temporal_window",
    "valid_observations","valid_pixels","std_dev","p05","p95","reference_allowed","score_eligible"
]]

metric_qa = qa_metrics(metric_results)
print("Automated metric QA:")
display(metric_qa)


### Interpretation guardrail

A valid EO measurement is not automatically a scored ecological indicator. The output keeps the **raw value, reference value, intactness, concern, QA/provenance and score eligibility** separate.

The common pillars are:
- **C1 Extent**
- **C2 Vegetation**
- **C3 Fauna**
- **C4 Pressure**

C3 Fauna must come from appropriate biodiversity evidence such as field surveys, acoustics, eDNA or another validated source; EO water proxies do not manufacture fauna evidence.


## 7. Tier-1 / Tier-2 benchmarking

In [ ]:

# Optional Tier-1 inputs.
# Set ONE of these to a local file path after uploading it to Colab:
# TIER1_KML = "/content/reference_lake.kml"
# TIER1_CSV = "/content/tier1_reference.csv"
TIER1_KML = None
TIER1_CSV = None

if TIER1_KML:
    cfg.reference.tier1_reference_kml = TIER1_KML
if TIER1_CSV:
    cfg.reference.tier1_reference_csv = TIER1_CSV

print("Tier-1 KML:", cfg.reference.tier1_reference_kml)
print("Tier-1 CSV:", cfg.reference.tier1_reference_csv)
print("Tier-2 candidate:", "context ring" if cfg.reference.tier2_enabled else "disabled")


In [ ]:

from darukaa_adaptive.benchmark import ReferenceEngine, benchmark_dataframe

reference_engine = ReferenceEngine(cfg, metrics_engine)
tier1_geometry = reference_engine.build_tier1_geometry(cfg.reference.tier1_reference_kml)
tier2_geometry, tier2_status = reference_engine.build_tier2_candidate_geometry(
    domains["context"], BASELINE_START, BASELINE_END
)
print("Tier-2 candidate status:", tier2_status)

benchmarks = reference_engine.build(
    metric_results,
    domains["boundary"],
    tier1_geometry=tier1_geometry,
    tier2_geometry=tier2_geometry,
    baseline_start=BASELINE_START,
    baseline_end=BASELINE_END,
)
benchmark_df = benchmark_dataframe(benchmarks)
benchmark_df


**Reference rule:** Tier-1 is preferred when available. Tier-2 is a candidate regional/context benchmark and is not silently treated as a pristine control.

For a metric to become score-eligible, the selected reference must be **explicitly approved for scoring**. Once approved, the metric is converted to a direction-aware **0–100 intactness score**, then assigned the fixed five-band concern level. Metric-specific literature thresholds are not required for this product scoring layer.


## 8. Concern scoring, pillar scores and overall SoN

In [ ]:

from darukaa_adaptive.scoring import build_scorecard

metric_concern_df, pillar_df, overall = build_scorecard(metric_results, benchmarks, cfg)
print("Metric concern scorecard:")
display(metric_concern_df)
print("Pillar scorecard:")
display(pillar_df)
print("Overall score:")
print(overall)


### Optional field / acoustic / terrestrial metrics

The same scoring engine can accept external observations without changing the aquatic metric code. A field metric must provide its pillar, direction and an approved comparable reference.

Required columns:
`metric, pillar, raw_value, direction, reference_value`

Optional:
`units, reference_type, reference_level, reference_approved_for_scoring, status, notes`


In [ ]:
# Example only — do not run until field/reference values are available.
# from darukaa_adaptive.scoring import score_external_observations
# field_obs = pd.DataFrame([
#     {
#         "metric": "species_richness",
#         "pillar": "C3_fauna",
#         "raw_value": 40,
#         "direction": "higher_is_better",
#         "reference_value": 50,
#         "reference_type": "field_reference",
#         "reference_level": "tier1",
#         "reference_approved_for_scoring": True,
#     }
# ])
# field_metric_concern, field_pillars, field_overall = score_external_observations(field_obs, cfg)
# display(field_metric_concern)
# display(field_pillars)
# print(field_overall)


### Scoring architecture

The production scoring sequence is:

`raw value → approved comparable reference → intactness (0–100) → indicator concern → pillar geometric mean → pillar concern → overall geometric mean → overall concern`

The same five fixed bands are used at indicator, pillar and overall level.

**Pillar:** geometric mean of continuous 0–100 intactness values for score-eligible indicators.  
**Overall SoN:** geometric mean of C1, C2, C3 and C4 when all four required pillars are represented.

The scorecard also identifies the **limiting indicator** within each pillar. The overall result identifies the **limiting pillar** and the limiting indicator within that pillar. Concern labels are never averaged.

For Nandoshi's EO-only run, the overall SoN will normally remain unavailable until C3 Fauna and any other missing pillar evidence are supplied.


## 9. Readiness assessment

In [ ]:

from darukaa_adaptive.readiness import assess_readiness

readiness = assess_readiness(
    cfg,
    area_ha(geom),
    metric_results,
    benchmarks,
    overall,
)
readiness


## 10. Write the complete auditable output package

In [ ]:

from darukaa_adaptive.report import write_assessment

paths = write_assessment(
    cfg.output_dir,
    cfg,
    SITE_FILE,
    area_ha(geom),
    domains,
    metric_results,
    monthly,
    readiness,
    benchmarks=benchmarks,
    scored_df=metric_concern_df,
    pillar_df=pillar_df,
    overall=overall,
    landcover=metrics_engine.landcover_composition(domains["boundary"], BASELINE_START, BASELINE_END),
    metric_qa=metric_qa,
    extra_manifest={"notebook_git_commit": GIT_SHA},
)
print("Outputs written:")
for k, v in paths.items():
    print(f"  {k}: {v}")


## 11. Freeze the Year-0 baseline artifact

In [ ]:

from shutil import copy2
from pathlib import Path

baseline_copy = Path(cfg.output_dir) / "baseline_metric_scorecard.csv"
copy2(Path(paths["metric_scorecard"]), baseline_copy)
print("✓ Year-0 baseline saved as:", baseline_copy)


## 12. Future monitoring comparison

In [ ]:
# For Year-1+, point CURRENT_METRIC_CSV to the latest monitoring scorecard and BASELINE_FILE to the frozen Year-0 file.
BASELINE_FILE = str(Path(cfg.output_dir) / "baseline_metric_scorecard.csv")
CURRENT_METRIC_CSV = None  # e.g. "/content/year1/outputs/metric_scorecard.csv"

if CURRENT_METRIC_CSV:
    from darukaa_adaptive.trajectory import compare
    trajectory_df = compare(CURRENT_METRIC_CSV, BASELINE_FILE)
    display(trajectory_df)
else:
    print("Year-0 run complete. Set CURRENT_METRIC_CSV to a later monitoring scorecard for trajectory comparison.")



### Monitoring interpretation

Use the same seasonal window definition in every cycle. A positive/negative raw change is not automatically ecological recovery/degradation; interpretation depends on metric direction, hydrological context, field validation and the approved ecological reference framework.


## 13. Package outputs for download

In [ ]:

import shutil

archive_base = "darukaa_nandoshi_aquatic_outputs"
archive_path = shutil.make_archive(archive_base, "zip", cfg.output_dir)
print("✓ Output archive:", archive_path)


## Troubleshooting

**Import error after a GitHub update:** rerun the repository sync/install cell; it clears cached `darukaa_adaptive` modules and reinstalls from the current folder.

**`git pull --ff-only` stops:** the notebook detected local uncommitted changes. Commit/stash them or restart a clean Colab runtime. The notebook will not use `git reset --hard`.

**Earth Engine authentication fails:** confirm the Google account has access to the selected GEE project and rerun the authentication cell.

**No water detected:** inspect the configured Dynamic World image count, optical cloud threshold, selected period, and Sentinel-1 fallback status. Do not hand-draw a replacement water polygon inside the assessment workflow.

**Metric concern is `reference_not_approved_for_scoring`:** the benchmark exists, but the selected reference has not yet been approved in the profile. This is a deliberate scoring gate.

**Overall score is `None`:** inspect the C1/C2/C3/C4 pillar scorecard. The overall SoN requires the configured four-pillar evidence coverage; an EO-only lake run cannot create missing fauna evidence.

**Important:** the five concern bands are the declared Darukaa product convention, not universal ecological thresholds.
